# Multivariable Calculus

When a function depends on **two or more variables**, e.g. $f(x,y)$, its graph lives in 3D and calculus extends with **partial derivatives**, **gradients**, and **multiple integrals**.


## Functions of Two Variables — Surface Plots

A function $f : \mathbb{R}^2 \to \mathbb{R}$ maps each point $(x,y)$ to a height $z = f(x,y)$.

The graph is a **surface** in $\mathbb{R}^3$. Level sets $f(x,y) = k$ are the **contour lines** (like topographic maps).


In [1]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display

surfaces = {
    'paraboloid: x²+y²':  lambda X, Y: X**2 + Y**2,
    'saddle: x²−y²':      lambda X, Y: X**2 - Y**2,
    'wave: sin(x)cos(y)':  lambda X, Y: np.sin(X) * np.cos(Y),
    'ripple: sin(√(x²+y²))': lambda X, Y: np.sin(np.sqrt(X**2 + Y**2 + 0.01)),
}

def plot_surface(name, elev, azim):
    x = np.linspace(-4, 4, 80)
    y = np.linspace(-4, 4, 80)
    X, Y = np.meshgrid(x, y)
    Z = surfaces[name](X, Y)

    fig = plt.figure(figsize=(10, 4))
    ax3 = fig.add_subplot(121, projection='3d')
    ax3.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85, edgecolor='none')
    ax3.set(xlabel='x', ylabel='y', zlabel='z')
    ax3.view_init(elev=elev, azim=azim)
    ax3.set_title(name)

    ax2 = fig.add_subplot(122)
    cs = ax2.contourf(X, Y, Z, levels=20, cmap='viridis')
    plt.colorbar(cs, ax=ax2)
    ax2.set(xlabel='x', ylabel='y', aspect='equal', title='Contour map')
    plt.tight_layout(); plt.show()

wn = widgets.Dropdown(options=list(surfaces.keys()), description='Surface')
we = widgets.IntSlider(value=30, min=0, max=90, step=5, description='elevation')
wa = widgets.IntSlider(value=45, min=0, max=360, step=5, description='azimuth')
display(widgets.VBox([wn, we, wa]),
        widgets.interactive_output(plot_surface, {'name': wn, 'elev': we, 'azim': wa}))


Output()

## The Gradient

For a scalar field $f(x,y)$, the **gradient** is the vector of partial derivatives:

$$\nabla f = \left(\frac{\partial f}{\partial x},\; \frac{\partial f}{\partial y}\right)$$

The gradient points in the direction of **steepest ascent** and its magnitude is the rate of change in that direction.

Example: $f(x,y) = x^2 + y^2 \;\Rightarrow\; \nabla f = (2x,\; 2y)$.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display

def f_grad(x, y): return x**2 + y**2
def grad_f(x, y): return 2*x, 2*y

x = np.linspace(-4, 4, 80)
y = np.linspace(-4, 4, 80)
X, Y = np.meshgrid(x, y)
Z = f_grad(X, Y)

def plot_gradient(px, py):
    gx, gy = grad_f(px, py)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    cs = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
    plt.colorbar(cs, ax=ax)
    ax.quiver(px, py, gx, gy, color='red', scale=40, width=0.008,
              label=f'∇f = ({gx:.1f}, {gy:.1f})')
    ax.plot(px, py, 'ro', ms=6)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), aspect='equal', xlabel='x', ylabel='y')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    ax.set_title(f'f = x²+y²  |  ∇f({px:.1f},{py:.1f}) = ({gx:.1f}, {gy:.1f})  |  |∇f| = {np.hypot(gx,gy):.2f}')
    plt.tight_layout(); plt.show()

wpx = widgets.FloatSlider(value=1, min=-3.5, max=3.5, step=0.1, description='x₀')
wpy = widgets.FloatSlider(value=1, min=-3.5, max=3.5, step=0.1, description='y₀')
display(widgets.VBox([wpx, wpy]),
        widgets.interactive_output(plot_gradient, {'px': wpx, 'py': wpy}))


Output()

## Divergence

For a 2D vector field $\mathbf{F}(x,y) = (P, Q)$:

$$\text{div}\,\mathbf{F} = \nabla \cdot \mathbf{F} = \frac{\partial P}{\partial x} + \frac{\partial Q}{\partial y}$$

Divergence measures the **net outward flux** per unit area: $\text{div} > 0$ → source, $\text{div} < 0$ → sink, $\text{div} = 0$ → incompressible.

Example: $\mathbf{F} = (x, y) \;\Rightarrow\; \nabla\cdot\mathbf{F} = 1 + 1 = 2$ (uniform source).


In [3]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display

field_options = {
    'source: F=(x,y), div=2':    (lambda X,Y:(X,Y), lambda x,y: 2.0),
    'sink: F=(−x,−y), div=−2':   (lambda X,Y:(-X,-Y), lambda x,y: -2.0),
    'rotation: F=(−y,x), div=0': (lambda X,Y:(-Y,X), lambda x,y: 0.0),
    'non-uniform: F=(x²,y²), div=2x+2y': (lambda X,Y:(X**2,Y**2), lambda x,y: 2*x+2*y),
}

def plot_div(name):
    F_fn, div_fn = field_options[name]
    g = np.linspace(-3, 3, 16)
    X, Y = np.meshgrid(g, g)
    U, V = F_fn(X, Y)
    D = div_fn(X, Y)

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    mag = np.sqrt(U**2 + V**2); mag[mag==0] = 1
    q = ax.quiver(X, Y, U/mag, V/mag, D, cmap='coolwarm', clim=(-4,4), alpha=0.8)
    plt.colorbar(q, ax=ax, label='divergence')
    ax.set(xlim=(-3,3), ylim=(-3,3), aspect='equal', xlabel='x', ylabel='y')
    ax.grid(True, alpha=0.3)
    ax.set_title(name)
    plt.tight_layout(); plt.show()

wf = widgets.Dropdown(options=list(field_options.keys()), description='Field')
display(wf, widgets.interactive_output(plot_div, {'name': wf}))


Dropdown(description='Field', options=('source: F=(x,y), div=2', 'sink: F=(−x,−y), div=−2', 'rotation: F=(−y,x…

Output()

## Curl

For a 2D vector field $\mathbf{F} = (P, Q)$, the **scalar curl** (z-component of the 3D curl) is:

$$\text{curl}\,\mathbf{F} = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$$

Curl measures the **local rotation** (vorticity) of the field: positive → counter-clockwise, negative → clockwise.

In 3D, the full curl is: $\nabla \times \mathbf{F} = \left(\frac{\partial R}{\partial y} - \frac{\partial Q}{\partial z},\; \frac{\partial P}{\partial z} - \frac{\partial R}{\partial x},\; \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right)$.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display

curl_fields = {
    'rotation: F=(−y,x), curl=2':  (lambda X,Y:(-Y,X), lambda X,Y: np.full_like(X, 2.0)),
    'irrotational: F=(x,y), curl=0': (lambda X,Y:(X,Y), lambda X,Y: np.zeros_like(X)),
    'F=(y²,x²), curl=2x−2y': (lambda X,Y:(Y**2,X**2), lambda X,Y: 2*X - 2*Y),
}

def plot_curl(name):
    F_fn, curl_fn = curl_fields[name]
    g = np.linspace(-3, 3, 16)
    X, Y = np.meshgrid(g, g)
    U, V = F_fn(X, Y)
    C = curl_fn(X, Y)
    mag = np.sqrt(U**2 + V**2); mag[mag==0] = 1

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    q = ax.quiver(X, Y, U/mag, V/mag, C, cmap='PiYG', clim=(-4,4), alpha=0.8)
    plt.colorbar(q, ax=ax, label='curl (z-component)')
    ax.set(xlim=(-3,3), ylim=(-3,3), aspect='equal', xlabel='x', ylabel='y')
    ax.grid(True, alpha=0.3)
    ax.set_title(name)
    plt.tight_layout(); plt.show()

wf = widgets.Dropdown(options=list(curl_fields.keys()), description='Field')
display(wf, widgets.interactive_output(plot_curl, {'name': wf}))


## Double Integrals

The double integral

$$\iint_D f(x,y)\,dA$$

computes the **volume** under the surface $z = f(x,y)$ over a region $D$ in the $xy$-plane.

Over a rectangular region $[a,b]\times[c,d]$: $\displaystyle\int_a^b\!\int_c^d f(x,y)\,dy\,dx$.

Over a region bounded by curves, the inner limits become functions of the outer variable.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display

def plot_double_integral(example, elev, azim):
    fig = plt.figure(figsize=(10, 4.5))
    ax3 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122)

    if example == 'rectangular: ∫∫(x²+y²) over [0,2]×[0,3]':
        x = np.linspace(0, 2, 60); y = np.linspace(0, 3, 60)
        X, Y = np.meshgrid(x, y)
        Z = X**2 + Y**2
        vol = np.trapezoid(np.trapezoid(Z, y, axis=0), x)
        ax3.plot_surface(X, Y, Z, cmap='viridis', alpha=0.75, edgecolor='none')
        ax3.set_title(f'x²+y² over [0,2]×[0,3]
Volume ≈ {vol:.2f}')
        cs = ax2.contourf(X, Y, Z, levels=15, cmap='viridis')
        ax2.set_title('Integration domain')
    else:
        x = np.linspace(0, 1, 80); y = np.linspace(0, 1, 80)
        X, Y = np.meshgrid(x, y)
        mask = Y <= X
        Z = np.where(mask, X + Y, np.nan)
        ax3.plot_surface(X, Y, np.where(mask, X+Y, 0), cmap='viridis', alpha=0.75, edgecolor='none')
        # compute integral numerically
        val = 0
        dx = x[1]-x[0]
        for xi in x:
            yi = np.linspace(0, xi, 50)
            val += np.trapezoid(xi + yi, yi) * dx
        ax3.set_title(f'x+y over triangle 0≤y≤x≤1
Volume ≈ {val:.4f}')
        Z_plot = np.where(mask, X+Y, np.nan)
        cs = ax2.contourf(X, Y, Z_plot, levels=15, cmap='viridis')
        ax2.plot([0,1,1,0],[0,0,1,0],'r-', lw=2)
        ax2.set_title('Triangular domain')

    plt.colorbar(cs, ax=ax2)
    ax2.set(xlabel='x', ylabel='y', aspect='equal')
    ax3.set(xlabel='x', ylabel='y', zlabel='f(x,y)')
    ax3.view_init(elev=elev, azim=azim)
    plt.tight_layout(); plt.show()

we_ex = widgets.Dropdown(options=[
    'rectangular: ∫∫(x²+y²) over [0,2]×[0,3]',
    'triangular: ∫∫(x+y) over 0≤y≤x≤1'
], description='Example')
we = widgets.IntSlider(value=30, min=0, max=90, step=5, description='elevation')
wa = widgets.IntSlider(value=45, min=0, max=360, step=5, description='azimuth')
display(widgets.VBox([we_ex, we, wa]),
        widgets.interactive_output(plot_double_integral, {'example': we_ex, 'elev': we, 'azim': wa}))
